# **This notebook acts as the dataloader of MIMIC-III**

TESTING

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import seaborn as sns
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
import copy
from sklearn.metrics import mean_absolute_error, r2_score
import os
import glob

# **MIMIC-III**

In [ ]:
path_client_1 = "../../Datasets/mimic_iii_data"

In [ ]:
def load_csv(name):
    file_path = os.path.join(path_client_1, f"{name}.csv")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File {file_path} not found.")
    print(f"Loading {file_path} ...")
    return pd.read_csv(file_path, low_memory=False)

In [ ]:
dataframes = {}

dataframes["ADMISSIONS"] = load_csv("ADMISSIONS")
dataframes["DATETIMEEVENTS"] = load_csv("DATETIMEEVENTS")
dataframes["ICUSTAYS"] = load_csv("ICUSTAYS")
dataframes["INPUTEVENTS_CV"] = load_csv("INPUTEVENTS_CV")
dataframes["INPUTEVENTS_MV"] = load_csv("INPUTEVENTS_MV")
dataframes["LABEVENTS"] = load_csv("LABEVENTS")
dataframes["MICROBIOLOGYEVENTS"] = load_csv("MICROBIOLOGYEVENTS")
dataframes["NOTEEVENTS"] = load_csv("NOTEEVENTS")
dataframes["OUTPUTEVENTS"] = load_csv("OUTPUTEVENTS")
dataframes["PATIENTS"] = load_csv("PATIENTS")
dataframes["PROCEDUREEVENTS_MV"] = load_csv("PROCEDUREEVENTS_MV")
dataframes["SERVICES"] = load_csv("SERVICES")

# print("Loaded:", list(dataframes.keys()))

In [ ]:
dataframes["ICUSTAYS"]

In [ ]:
OBS_WINDOW = pd.Timedelta(hours=24)
BIN_SIZE   = pd.Timedelta(hours=1)
id_col = "ICUSTAY_ID"
client_1 = dataframes["ICUSTAYS"][["ICUSTAY_ID", "SUBJECT_ID", "INTIME", "OUTTIME"]].copy()

client_1["INTIME"]  = pd.to_datetime(client_1["INTIME"],  errors="coerce")
client_1["OUTTIME"] = pd.to_datetime(client_1["OUTTIME"], errors="coerce")

stays_for_event_windowing = client_1[[id_col, "INTIME"]].copy()
stays_for_event_windowing = stays_for_event_windowing.rename(columns={"INTIME":"admit_time"})

print(client_1.shape)                  
print(stays_for_event_windowing.shape)

In [ ]:
merge_log = []

for name, df in dataframes.items():
    # We only process tables that have an ICUSTAY_ID
    if id_col not in df.columns:
        print(f"Skipping {name}: no {id_col}")
        continue

    print(f"\nProcessing {name}.csv")
    df = df.copy()

    # 1) Find & parse eventtime
    # Look for any column with 'time' in name (datetime)…
    dt_cols  = [c for c in df.columns if "time" in c.lower()]
    off_cols = [c for c in df.columns if "offset" in c.lower()]

    if dt_cols:
        first_dt = dt_cols[0]
        df[first_dt] = pd.to_datetime(df[first_dt], errors="coerce")
        df = df.merge(stays_for_event_windowing, on=id_col, how="left")
        df["eventtime"] = df[first_dt]

    elif off_cols:
        first_off = off_cols[0]
        df = df.merge(stays_for_event_windowing, on=id_col, how="left")
        df["eventtime"] = (
            df["admit_time"] 
            + pd.to_timedelta(df[first_off].astype(float), unit="m")
        )
    else:
        print(f"  No datetime or offset in {name}, skipping")
        continue

    # 2) Dilter to first 24 h and bin into hours
    before = len(df)
    df = df.loc[
        (df["eventtime"] >= df["admit_time"]) &
        (df["eventtime"] <  df["admit_time"] + OBS_WINDOW)
    ].copy()
    after = len(df)
    print(f"  Kept {after}/{before} rows in first 24 h")

    df["hour_from_admit"] = (
        (df["eventtime"] - df["admit_time"])
        .dt.total_seconds()
        .floordiv(BIN_SIZE.total_seconds())
        .astype(int)
    )

    # 3a) Numeric aggregation
    numeric_cols = df.select_dtypes(include="number")\
                     .columns.difference([id_col,"hour_from_admit"])
    num_wide = None
    if len(numeric_cols):
        grp = df.groupby([id_col,"hour_from_admit"])[numeric_cols] \
                .agg(["mean","std","min","max","count"])
        num_wide = grp.unstack(level="hour_from_admit", fill_value=np.nan)
        num_wide.columns = [
            f"{name}_{orig}_{stat}_h{hour}"
            for orig, stat, hour in num_wide.columns
        ]
        num_wide = num_wide.reset_index()

    # 3b) Categorical aggregation
    cat_cols = df.select_dtypes(include=["object","category","bool"])\
                 .columns.difference([id_col,"hour_from_admit"])
    cat_wide = None
    if len(cat_cols):
        grp = df.groupby([id_col,"hour_from_admit"])[cat_cols] \
                .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
        cat_wide = grp.unstack(level="hour_from_admit", fill_value=pd.NA)
        cat_wide.columns = [
            f"{name}_{orig}_mode_h{hour}"
            for orig, hour in cat_wide.columns
        ]
        cat_wide = cat_wide.reset_index()

    # 4) Combine and prune all-NaN
    parts = [t for t in (num_wide, cat_wide) if t is not None]
    if not parts:
        print(f"  Skipping {name}: nothing to aggregate")
        continue

    hourly_wide = parts[0]
    for t in parts[1:]:
        hourly_wide = hourly_wide.merge(t, on=id_col, how="outer")

    keep = [c for c in hourly_wide.columns
            if c == id_col or not hourly_wide[c].isna().all()]
    hourly_wide = hourly_wide[keep]

    # 5) Merge into client_1
    overlap = set(hourly_wide.columns) & set(client_1.columns) - {id_col}
    if overlap:
        hourly_wide = hourly_wide.drop(columns=list(overlap))

    client_1 = client_1.merge(hourly_wide, on=id_col, how="left")
    merge_log.append((name, "hourly"))

print("\nMIMIC merged shape:", client_1.shape)
print("Merge log:", merge_log)

In [ ]:
patients = dataframes["PATIENTS"][["SUBJECT_ID","GENDER","DOB","DOD"]].copy()

patients["DOB"] = pd.to_datetime(patients["DOB"], errors="coerce")
patients["DOD"] = pd.to_datetime(patients["DOD"], errors="coerce")

client_1 = client_1.merge(patients, on="SUBJECT_ID", how="left")

print("With demographics:", client_1.shape)
print(client_1[["SUBJECT_ID","GENDER","DOB","DOD"]].head())

In [ ]:
client_1.to_csv("client_1_ICUSTAY_raw_hour.csv", index=False)

In [ ]:
client_1

In [ ]:
client_1.to_csv("client_1_ICUSTAY_raw.csv", index=False)

In [ ]:
# Check all column names:
client_1.columns